# SmartNav YOLO26n Edge Training on T4 GPU (33 Classes - 416x416 Mobile Optimized)

## Visual Obstacle Detection & Indoor Navigation for Wearable Assistive Glasses

### Workflow
1. Dataset `datasets/smartnav_v1_33cls_20260920` is already configured in `datasets/data.yaml`.
2. Zip `datasets/smartnav_v1_33cls_20260920/train`, `val`, and `data.yaml` as `datasets_colab.zip`:
   ```powershell
   Compress-Archive -Path datasets\smartnav_v1_33cls_20260920\train,datasets\smartnav_v1_33cls_20260920\val,datasets\smartnav_v1_33cls_20260920\data.yaml -DestinationPath datasets_colab.zip -Force
   ```
3. Upload `datasets_colab.zip` to Google Drive root (`MyDrive/datasets_colab.zip`).
4. Run cells below:
   - **Mount & Extract**
   - **Rare-Class Oversampling** (boosts manhole, rickshaw, van, stop_sign, cow)
   - **Train YOLO26n** (imgsz=416, epochs=120, patience=20)
   - **Export INT8 TFLite** for on-device Android phone deployment


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Extract dataset from Drive
Looks for `datasets_colab.zip` first, then falls back to `datasets.zip`.
Handles Windows backslash paths, nested `datasets/` zip roots, stale label caches, and Windows paths inside `data.yaml`.


In [ ]:
import zipfile, shutil, os, re

DATASET_ROOT = '/content/datasets'
ZIP_CANDIDATES = [
    '/content/drive/MyDrive/datasets_colab.zip',
    '/content/drive/MyDrive/datasets.zip',
]

zip_path = next((p for p in ZIP_CANDIDATES if os.path.exists(p)), None)
if zip_path is None:
    raise FileNotFoundError('Upload datasets_colab.zip to MyDrive first.')

# Clean slate so old label caches and stale files cannot leak into training.
shutil.rmtree(DATASET_ROOT, ignore_errors=True)
os.makedirs(DATASET_ROOT, exist_ok=True)

with zipfile.ZipFile(zip_path) as z:
    for name in z.namelist():
        normal = name.replace('\\', '/')
        z.extract(name, DATASET_ROOT)
        if normal != name:
            src = os.path.join(DATASET_ROOT, name)
            dst = os.path.join(DATASET_ROOT, normal)
            if os.path.exists(src) and not os.path.exists(dst):
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                os.rename(src, dst)

# If the zip contains a top-level datasets/ folder, flatten it into /content/datasets.
nested_root = os.path.join(DATASET_ROOT, 'datasets')
if os.path.exists(os.path.join(nested_root, 'data.yaml')) and not os.path.exists(os.path.join(DATASET_ROOT, 'data.yaml')):
    for item in os.listdir(nested_root):
        src = os.path.join(nested_root, item)
        dst = os.path.join(DATASET_ROOT, item)
        if os.path.isdir(dst):
            shutil.rmtree(dst)
        elif os.path.exists(dst):
            os.remove(dst)
        shutil.move(src, dst)
    shutil.rmtree(nested_root, ignore_errors=True)

# Remove any old Ultralytics caches if this runtime had a previous dataset.
for cache_path in [
    os.path.join(DATASET_ROOT, 'train', 'labels.cache'),
    os.path.join(DATASET_ROOT, 'val', 'labels.cache'),
]:
    if os.path.exists(cache_path):
        os.remove(cache_path)

# Fix data.yaml paths (Windows -> Colab).
yaml_path = os.path.join(DATASET_ROOT, 'data.yaml')
if not os.path.exists(yaml_path):
    raise FileNotFoundError(f'{yaml_path} not found after extracting {zip_path}')

with open(yaml_path, encoding='utf-8') as f:
    content = f.read()
content = content.replace('\\', '/')
content = re.sub(r'[A-Za-z]:/.+?/datasets/', '/content/datasets/', content)
content = re.sub(r'train:\s*.*', 'train: /content/datasets/train/images', content)
content = re.sub(r'val:\s*.*', 'val: /content/datasets/val/images', content)
with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(content)

required = [
    os.path.join(DATASET_ROOT, 'train', 'images'),
    os.path.join(DATASET_ROOT, 'train', 'labels'),
    os.path.join(DATASET_ROOT, 'val', 'images'),
    os.path.join(DATASET_ROOT, 'val', 'labels'),
    yaml_path,
]
missing = [p for p in required if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f'Missing dataset paths after extract: {missing}')

print('Using zip:', zip_path)
print('Extracted:', sorted(os.listdir(DATASET_ROOT)))
print('Train images:', len(os.listdir(os.path.join(DATASET_ROOT, 'train', 'images'))))
print('Val images:', len(os.listdir(os.path.join(DATASET_ROOT, 'val', 'images'))))


---
## Install dependencies

In [ ]:
!pip install -q ultralytics requests

---
## Train (fresh start)
Saves checkpoint to Drive after every epoch. If disconnected, use the **Resume** cell below.

In [ ]:
# Class-weighted oversampling for rare classes before training
# Duplicates rare class image & label entries so they appear more frequently per epoch
import os, shutil
from pathlib import Path
from collections import Counter

DATASET_ROOT = Path('/content/datasets')
train_lbl_dir = DATASET_ROOT / 'train' / 'labels'
train_img_dir = DATASET_ROOT / 'train' / 'images'

# Rare class IDs to oversample: manhole(11), rickshaw(6), van(7), stop_sign(14), cow(19)
OVERSAMPLE_CONFIG = {
    11: 8,  # manhole: multiply by 8
    6:  3,  # rickshaw: multiply by 3
    7:  4,  # van: multiply by 4
    14: 3,  # stop_sign: multiply by 3
    19: 2,  # cow: multiply by 2
}

added_count = 0
for lbl_file in list(train_lbl_dir.glob('*.txt')):
    stem = lbl_file.stem
    if '_os_' in stem:
        continue
    cids = set()
    with open(lbl_file, 'r', errors='ignore') as fp:
        for line in fp:
            p = line.split()
            if p:
                try:
                    cids.add(int(p[0]))
                except ValueError:
                    pass
    max_mult = max([OVERSAMPLE_CONFIG.get(cid, 1) for cid in cids], default=1)
    if max_mult > 1:
        # Find source image
        img_path = None
        for ext in ['.jpg', '.png', '.jpeg']:
            cand = train_img_dir / f'{stem}{ext}'
            if cand.exists():
                img_path = cand
                break
        if img_path:
            for m in range(1, max_mult):
                os_stem = f'{stem}_os_{m}'
                shutil.copy2(lbl_file, train_lbl_dir / f'{os_stem}.txt')
                shutil.copy2(img_path, train_img_dir / f'{os_stem}{img_path.suffix}')
                added_count += 1

print(f'Oversampling complete: added {added_count} rare-class training samples.')
print(f'Total train images after oversampling: {len(os.listdir(train_img_dir))}')


In [ ]:
import shutil, os
from ultralytics import YOLO

CHECKPOINT_DIR = '/content/drive/MyDrive/vqa_training'
LAST_CKPT = f'{CHECKPOINT_DIR}/last.pt'
BEST_CKPT  = f'{CHECKPOINT_DIR}/best.pt'
RUNS_DIR   = '/content/runs/smartnav_yolo26n'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def save_checkpoint(trainer):
    src_last = f'{trainer.save_dir}/weights/last.pt'
    src_best = f'{trainer.save_dir}/weights/best.pt'
    if os.path.exists(src_last):
        shutil.copy(src_last, LAST_CKPT)
    if os.path.exists(src_best):
        shutil.copy(src_best, BEST_CKPT)
    print(f'  -> Checkpoints saved to Drive ({CHECKPOINT_DIR})')

# Use YOLO26n specifically (NMS-free end-to-end head for mobile edge deployment)
model = YOLO('yolo26n.pt')
model.add_callback('on_train_epoch_end', save_checkpoint)

# Training config: imgsz=416, epochs=120, patience=20, mobile augmentation
results = model.train(
    data='/content/datasets/data.yaml',
    epochs=120,
    patience=20,
    imgsz=416,
    batch=32,
    optimizer='AdamW',
    lr0=0.001,
    device=0,
    workers=4,
    seed=42,
    cos_lr=True,
    warmup_epochs=3,
    amp=True,
    mosaic=1.0,
    mixup=0.1,
    degrees=10.0,
    fliplr=0.5,
    project='/content/runs',
    name='smartnav_yolo26n',
    exist_ok=True
)


---
## Resume after disconnection
Run this if Colab disconnected mid-training. Make sure you ran **Mount** and **Extract** cells first.
This reads the checkpoint from Drive and continues training from the last completed epoch.

In [ ]:
import shutil, os
from ultralytics import YOLO

CHECKPOINT_DIR = '/content/drive/MyDrive/vqa_training'
LAST_CKPT = f'{CHECKPOINT_DIR}/last.pt'
RUNS_DIR  = '/content/runs/smartnav_yolo26n'

if not os.path.exists(LAST_CKPT):
    raise FileNotFoundError(f'{LAST_CKPT} not found. Run Train cell first.')

os.makedirs(f'{RUNS_DIR}/weights', exist_ok=True)
shutil.copy(LAST_CKPT, f'{RUNS_DIR}/weights/last.pt')

def save_checkpoint(trainer):
    src_last = f'{trainer.save_dir}/weights/last.pt'
    src_best = f'{trainer.save_dir}/weights/best.pt'
    if os.path.exists(src_last):
        shutil.copy(src_last, LAST_CKPT)
    if os.path.exists(src_best):
        shutil.copy(src_best, f'{CHECKPOINT_DIR}/best.pt')
    print(f'  -> Checkpoints saved to Drive')

model = YOLO(f'{RUNS_DIR}/weights/last.pt')
model.add_callback('on_train_epoch_end', save_checkpoint)
model.train(resume=True)


---
## Save best model to Drive
Run after training completes (all epochs). Copies `best.pt` to Drive.

In [ ]:
import shutil, os
CHECKPOINT_DIR = '/content/drive/MyDrive/vqa_training'
BEST_CKPT = f'{CHECKPOINT_DIR}/best.pt'

best_local = '/content/runs/vqa_expanded/weights/best.pt'
if os.path.exists(best_local):
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    shutil.copy(best_local, BEST_CKPT)
    shutil.copy(best_local, '/content/drive/MyDrive/best.pt')
    print(f'Saved best.pt to Drive ({os.path.getsize(best_local)/1024/1024:.1f} MB)')
elif os.path.exists(BEST_CKPT):
    shutil.copy(BEST_CKPT, '/content/drive/MyDrive/best.pt')
    print(f'Copied best.pt from checkpoint dir to MyDrive root')
else:
    print('best.pt not found. Training may not have completed.')

In [ ]:
# Export to TFLite INT8 for On-Device Android Phone Deployment
from ultralytics import YOLO
import shutil, os

CHECKPOINT_DIR = '/content/drive/MyDrive/vqa_training'
BEST_CKPT = f'{CHECKPOINT_DIR}/best.pt'

print('Loading best trained YOLO26n model...')
model = YOLO(BEST_CKPT)

print('Exporting to TFLite with INT8 quantization at imgsz=416...')
tflite_path = model.export(
    format='tflite',
    int8=True,
    data='/content/datasets/data.yaml',
    imgsz=416
)

print(f'TFLite export complete: {tflite_path}')
dst_drive = '/content/drive/MyDrive/smartnav_yolo26n_int8.tflite'
shutil.copy(tflite_path, dst_drive)
print(f'Saved INT8 model to Drive: {dst_drive} ({os.path.getsize(dst_drive)/1024/1024:.2f} MB)')
